# Data Engineering Pipeline - Blood Pressure Dataset

In [13]:
import pandas as pd

In [14]:
# Load dataset
df = pd.read_csv("blood_pressure_global_dataset.csv")
df.columns = df.columns.str.lower().str.strip()
df.head()

,patient_id,year,country,who_region,income_level,currency,iso2_country_code,age,age_group,sex,...,bp_category,is_hypertensive,hypertension_aware,on_treatment,bp_controlled,country_htn_prevalence_pct,measurement_time,measurement_arm,measurement_setting,measurement_device
0,BP-00001,2000,Papua New Guinea,Western Pacific,Lower-Middle Income,PGK,pg,34,Adult (30-39),Male,...,Normal,0,NaN,No,No,38,Morning,Right,Home,Mercury Sphygmomanometer
1,BP-00002,2022,Uganda,Africa,Low Income,UGX,ug,3,Early Childhood (1-5),Female,...,Hypotension (Low),0,NaN,No,No,47,Morning,Left,Pharmacy,Wrist Digital
2,BP-00003,2024,Kenya,Africa,Lower-Middle Income,KES,ke,29,Young Adult (19-29),Male,...,Elevated,0,NaN,No,No,42,Morning,Right,Clinical/Hospital,Automated Oscillometric
3,BP-00004,2014,Thailand,South-East Asia,Upper-Middle Income,THB,th,53,Middle-Aged Senior (50-59),Male,...,Hypertension Stage 2,1,No,No,No,32,Afternoon,Right,Community Screening,Automated Oscillometric
4,BP-00005,2001,India,South-East Asia,Lower-Middle Income,INR,in,11,Early Adolescence (11-15),Female,...,Normal,0,NaN,No,No,29,Evening,Right,Home,Automated Oscillometric


print(df.columns)

In [15]:
# Check structure
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 34 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   patient_id                  8000 non-null   str    
 1   year                        8000 non-null   int64  
 2   country                     8000 non-null   str    
 3   who_region                  8000 non-null   str    
 4   income_level                8000 non-null   str    
 5   currency                    8000 non-null   str    
 6   iso2_country_code           8000 non-null   str    
 7   age                         8000 non-null   int64  
 8   age_group                   8000 non-null   str    
 9   sex                         8000 non-null   str    
 10  bmi                         8000 non-null   float64
 11  bmi_category                8000 non-null   str    
 12  smoking_status              8000 non-null   str    
 13  alcohol_use                 3748 non-null   

In [16]:
# Data Cleaning
df = df.dropna()
df = df.drop_duplicates()

df.shape

(2438, 34)

In [17]:
print(df.columns)

Index(['patient_id', 'year', 'country', 'who_region', 'income_level',
       'currency', 'iso2_country_code', 'age', 'age_group', 'sex', 'bmi',
       'bmi_category', 'smoking_status', 'alcohol_use', 'physical_activity',
       'diet_salt_intake', 'stress_level', 'diabetes',
       'family_hx_hypertension', 'systolic_bp_mmhg', 'diastolic_bp_mmhg',
       'pulse_pressure_mmhg', 'mean_arterial_pressure', 'heart_rate_bpm',
       'bp_category', 'is_hypertensive', 'hypertension_aware', 'on_treatment',
       'bp_controlled', 'country_htn_prevalence_pct', 'measurement_time',
       'measurement_arm', 'measurement_setting', 'measurement_device'],
      dtype='str')


In [18]:
# Select relevant columns (edit if needed)
columns_needed = [
    'country',
    'year',
    'systolic_bp_mmhg',
    'diastolic_bp_mmhg',
    'bmi',
    'smoking_status',
    'bp_category'
]

df = df[columns_needed]

df.head()

,country,year,systolic_bp_mmhg,diastolic_bp_mmhg,bmi,smoking_status,bp_category
3,Thailand,2014,150.1,76.2,23.7,Ex-Smoker,Hypertension Stage 2
12,Egypt,2018,113.6,81.7,20.8,Non-Smoker,Hypertension Stage 1
16,Trinidad & Tobago,2025,161.2,83.9,24.5,Current Smoker,Hypertension Stage 1
19,Kenya,2018,137.3,72.7,25.0,Non-Smoker,Hypertension Stage 1
24,Portugal,2011,179.1,73.4,18.1,Current Smoker,Hypertension Stage 2


In [19]:
# Add category column
def kategori_bp(systolic):
    if systolic >= 140:
        return "Hypertension"
    elif systolic >= 120:
        return "Prehypertension"
    else:
        return "Normal"

df['category'] = df['systolic_bp_mmhg'].apply(kategori_bp)

df.head()

,country,year,systolic_bp_mmhg,diastolic_bp_mmhg,bmi,smoking_status,bp_category,category
3,Thailand,2014,150.1,76.2,23.7,Ex-Smoker,Hypertension Stage 2,Hypertension
12,Egypt,2018,113.6,81.7,20.8,Non-Smoker,Hypertension Stage 1,Normal
16,Trinidad & Tobago,2025,161.2,83.9,24.5,Current Smoker,Hypertension Stage 1,Hypertension
19,Kenya,2018,137.3,72.7,25.0,Non-Smoker,Hypertension Stage 1,Prehypertension
24,Portugal,2011,179.1,73.4,18.1,Current Smoker,Hypertension Stage 2,Hypertension


In [20]:
# Export cleaned data
df.to_excel("cleaned_blood_pressure.xlsx", index=False)

print("Data exported successfully!")

Data exported successfully!


In [23]:
# Create data dictionary
data_dict = pd.DataFrame({
    "column": df.columns,
    "data_type": df.dtypes.astype(str),
    "description": [
        "Nama negara",
        "Tahun data",
        "Tekanan darah sistolik",
        "Tekanan darah diastolik",
        "BMI",
        "Status perokok",
        "Stage tekanan darah",
        "Kategori tekanan darah"
    ]
})

data_dict.to_excel("data_dictionary.xlsx", index=False)

# Export ke file SQL
file_name = "blood_pressure_clean.sql"

with open(file_name, "w") as f:
    # CREATE TABLE
    f.write("CREATE TABLE blood_pressure_clean (\n")
    
    for col in df.columns:
        f.write(f"    {col} TEXT,\n")
    
    f.write(");\n\n")

    # INSERT DATA
    for _, row in df.iterrows():
        values = []
        for val in row:
            val = str(val).replace("'", "''")
            values.append(f"'{val}'")
        
        values_str = ", ".join(values)
        f.write(f"INSERT INTO blood_pressure_clean VALUES ({values_str});\n")

print("✅ File SQL berhasil dibuat!")

data_dict

✅ File SQL berhasil dibuat!


,column,data_type,description
country,country,str,Nama negara
year,year,int64,Tahun data
systolic_bp_mmhg,systolic_bp_mmhg,float64,Tekanan darah sistolik
diastolic_bp_mmhg,diastolic_bp_mmhg,float64,Tekanan darah diastolik
bmi,bmi,float64,BMI
smoking_status,smoking_status,str,Status perokok
bp_category,bp_category,str,Stage tekanan darah
category,category,str,Kategori tekanan darah
